# 수정 표본의 대표성 보조 진단
고정 계획과 12번 데이터셋을 읽어 중심점 500m·가용률 100% 표본의 구성을 점검한다. 최소 업종별 군집 수를 적용하기 전의 **공통 결과 가용 표본**과 **상권×업종별 2분기 이상 반복 표본**을 분리한다. 실제 회귀 표본이나 업종별 추정 가능성을 대신하지 않는다.

등록 후보 0, 선택된 관측기간의 예정 시작 0, 고정 연구기간 전체의 예정 시작 0은 다르다. 이 노트북은 관측된 노출과 선택 구조만 기술한다. 새로운 회귀·p값·효과 방향을 계산하지 않으며 계획·원자료·12·13번 출력은 수정하지 않는다.

## 1. 고정 입력과 변경 금지 파일 확인
12번 완료 명세가 가리키는 패널·노출·중심점 파일 해시와 계획 해시를 확인한다. 13번 결과는 읽어 해석하지 않고 파일 해시만 전후 대조한다.

In [1]:
from pathlib import Path
import hashlib,json
import numpy as np
import pandas as pd
from IPython.display import display,Markdown
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'final/metadata/pipeline.json').is_file())
OUT=ROOT/'reports/final/revised_quarterly';OUT.mkdir(parents=True,exist_ok=True)
inputs={}
def register(relative,expected=None):
    path=ROOT/relative;digest=hashlib.sha256(path.read_bytes()).hexdigest()
    if expected is not None:assert digest==expected,relative
    if relative not in inputs:inputs[relative]={'path':relative,'sha256_before':digest,'expected_sha256':expected}
    return path
summary12=json.loads(register('reports/final/revised_quarterly/12_dataset_summary.json').read_text(encoding='utf-8'))
assert summary12['status']=='complete' and summary12['validation_all_passed'] and summary12['all_inputs_unchanged']
plan=json.loads(register(summary12['plan_path'],summary12['plan_sha256']).read_text(encoding='utf-8'))
for relative,digest in plan['protected_source_sha256'].items():register(relative,digest)
for relative,record in summary12['outputs'].items():register(relative,record['sha256'])
for directory in [ROOT/'data/analysis/final/revised_quarterly',OUT]:
    for path in directory.glob('13_*'):
        if path.is_file():register(path.relative_to(ROOT).as_posix())
for path in (ROOT/'final/notebooks').glob('1[23]_*.ipynb'):register(path.relative_to(ROOT).as_posix())
dtype={'area_id':str,'industry_id':str,'quarter':str}
panel=pd.read_csv(ROOT/summary12['panel_path'],dtype=dtype,float_precision='round_trip')
aqs=pd.read_csv(ROOT/'data/analysis/final/revised_quarterly/area_quarter_exposure.csv',dtype=dtype,float_precision='round_trip')
areas=pd.read_csv(ROOT/'data/analysis/final/revised_quarterly/area_centroids.csv',dtype=dtype,float_precision='round_trip')
aq=aqs.loc[aqs.spatial_definition.eq('centroid500')].merge(areas[['area_id','area_name','district']],on='area_id',validate='many_to_one')
assert len(panel)==18120 and len(aq)==1812 and not aq.duplicated(['area_id','quarter']).any()

## 2. 후보 0과 모든 선택 관측기간의 예정 시작 0
공통 결과 가용 표본은 총매출 양수·야간매출 비결측 및 0 이상이다. 가용률 100%를 적용한 뒤 상권×업종 2분기 이상 여부를 별도로 집계한다. 등록 후보가 없는 행은 고정 공개등록·공식 시설 연결 범위의 0이며 실제 공연 부재를 보장하지 않는다.

In [2]:
common_mask=panel.total_sales.gt(0)&panel.night_sales.notna()&panel.night_sales.ge(0)
base=panel.loc[common_mask & panel.availability_pct_centroid500.ge(100)].copy()
counts=base.groupby(['area_id','industry_id']).size()
repeat=base.set_index(['area_id','industry_id']).loc[lambda frame:frame.index.isin(counts[counts.ge(2)].index)].reset_index()
assert len(base)==summary12['sample_availability'][0]['common_outcome_rows']
assert len(repeat)==summary12['sample_availability'][0]['repeated_entity_rows']
selection_rows=[];entity_rows=[]
for label,frame in [('common_outcome_before_repeat_filter',base),('repeat_ge2_before_min_cluster_filter',repeat)]:
    entity=frame.groupby(['area_id','industry_id']).agg(n_observed_quarters=('quarter','size'),
        max_scheduled_start=('known_scheduled_start_count_centroid500','max'),
        min_scheduled_start=('known_scheduled_start_count_centroid500','min'))
    entity['all_selected_observations_zero']=entity.max_scheduled_start.eq(0)
    entity['selected_exposure_varies']=entity.max_scheduled_start.ne(entity.min_scheduled_start)
    zero_entity_rows=int(entity.loc[entity.all_selected_observations_zero,'n_observed_quarters'].sum())
    area=frame.groupby('area_id').agg(max_start=('known_scheduled_start_count_centroid500','max'),
        any_zero_candidate=('zero_registered_candidate_centroid500','any'))
    zero_candidate=frame.zero_registered_candidate_centroid500
    selection_rows.append(dict(sample=label,sales_rows=len(frame),areas=int(frame.area_id.nunique()),
        area_industry_entities=len(entity),zero_candidate_sales_rows=int(zero_candidate.sum()),
        zero_candidate_sales_row_pct=float(100*zero_candidate.mean()),
        areas_with_any_zero_candidate_observation=int(area.any_zero_candidate.sum()),
        areas_with_any_zero_candidate_observation_pct=float(100*area.any_zero_candidate.mean()),
        areas_all_selected_observations_zero=int(area.max_start.eq(0).sum()),
        areas_all_selected_observations_zero_pct=float(100*area.max_start.eq(0).mean()),
        entities_all_selected_observations_zero=int(entity.all_selected_observations_zero.sum()),
        entities_all_selected_observations_zero_pct=float(100*entity.all_selected_observations_zero.mean()),
        sales_rows_in_all_zero_entities=zero_entity_rows,sales_rows_in_all_zero_entities_pct=float(100*zero_entity_rows/len(frame)),
        entities_with_varying_selected_exposure=int(entity.selected_exposure_varies.sum())))
    entity_rows.append(entity.reset_index().assign(sample=label))
selection=pd.DataFrame(selection_rows);entities=pd.concat(entity_rows,ignore_index=True)
display(selection.T)

,0,1
sample,common_outcome_before_repeat_filter,repeat_ge2_before_min_cluster_filter
sales_rows,5042,5015
areas,118,114
area_industry_entities,597,570
zero_candidate_sales_rows,1869,1863
zero_candidate_sales_row_pct,37.068624,37.148554
areas_with_any_zero_candidate_observation,64,63
areas_with_any_zero_candidate_observation_pct,54.237288,55.263158
areas_all_selected_observations_zero,15,14
areas_all_selected_observations_zero_pct,12.711864,12.280702


**해석:** 모든 선택 관측기간 노출 0인 상권×업종은 해당 관측기간 안에서 노출의 시간 변화를 제공하지 않는다. 다른 계수나 공통 시간효과 추정에 쓰일 수 있으므로 이 행들이 쓸모없다고 해석하지 않는다. 판매자료가 없는 분기나 가용률로 빠진 분기까지 실제 공연이 0이었다고 확대 해석하지 않는다.

## 3. 완전 가용 표본과 제외 표본의 공연 밀집도
상권·분기를 한 번씩 세어 업종 10개로 반복된 행의 가중을 피한다. 후보 ID 수는 파싱 전 분모이며, 알려진 예정 시작회차는 파싱 가능한 부분 합계다. 미완전 표본의 부분 합계는 실제 회차 총량이나 누락 정도를 알려주지 않는다.

In [3]:
observed_aq=panel.loc[common_mask,['area_id','quarter']].drop_duplicates().assign(any_observed_sales=True)
aq=aq.merge(observed_aq,on=['area_id','quarter'],how='left',validate='one_to_one')
aq['any_observed_sales']=aq.any_observed_sales.fillna(False).astype(bool)
density_rows=[]
for population,part in [('all_area_quarters',aq),('positive_candidate_area_quarters',aq.loc[aq.candidate_id_count.gt(0)]),
    ('area_quarters_with_observed_sales',aq.loc[aq.any_observed_sales])]:
    for complete,subset in part.groupby('complete_schedule_coverage'):
        row=dict(population=population,complete_schedule_coverage=bool(complete),area_quarters=len(subset),
            areas=int(subset.area_id.nunique()),zero_candidate_area_quarters=int(subset.candidate_id_count.eq(0).sum()),
            median_availability_pct=float(subset.availability_pct.median()))
        for column in ['candidate_id_count','known_scheduled_start_count']:
            row.update({f'{column}_mean':float(subset[column].mean()),f'{column}_median':float(subset[column].median()),
                f'{column}_p90':float(subset[column].quantile(.9)),f'{column}_max':int(subset[column].max())})
        density_rows.append(row)
density=pd.DataFrame(density_rows)
display(density)

,population,complete_schedule_coverage,area_quarters,areas,zero_candidate_area_quarters,median_availability_pct,candidate_id_count_mean,candidate_id_count_median,candidate_id_count_p90,candidate_id_count_max,known_scheduled_start_count_mean,known_scheduled_start_count_median,known_scheduled_start_count_p90,known_scheduled_start_count_max
0,all_area_quarters,False,541,102,0,92.857143,72.929760,33.0,217.0,545,771.373383,184.0,2046.0,8110
1,all_area_quarters,True,1271,133,535,100.000000,7.575138,1.0,24.0,113,41.782848,2.0,118.0,720
2,positive_candidate_area_quarters,False,541,102,0,92.857143,72.929760,33.0,217.0,545,771.373383,184.0,2046.0,8110
3,positive_candidate_area_quarters,True,736,117,0,100.000000,13.081522,7.0,33.0,113,72.154891,23.5,238.0,720
4,area_quarters_with_observed_sales,False,474,90,0,93.333333,79.624473,35.5,236.6,545,836.681435,182.0,2436.6,8110
5,area_quarters_with_observed_sales,True,1119,118,478,100.000000,7.386953,1.0,24.0,113,39.127793,2.0,98.2,720


In [4]:
aq['candidate_density_band']=pd.cut(aq.candidate_id_count,bins=[-.5,.5,9.5,49.5,99.5,np.inf],labels=['0','1-9','10-49','50-99','100+'])
row_counts=panel.loc[common_mask].groupby(['area_id','quarter']).size().rename('provided_sales_rows').reset_index()
aq=aq.merge(row_counts,on=['area_id','quarter'],how='left',validate='one_to_one')
aq['provided_sales_rows']=aq.provided_sales_rows.fillna(0).astype(int)
aq['retained_sales_rows']=aq.provided_sales_rows.where(aq.complete_schedule_coverage,0)
density_bands=aq.groupby('candidate_density_band',observed=True).agg(area_quarters=('area_id','size'),
    complete_area_quarters=('complete_schedule_coverage','sum'),mean_availability_pct=('availability_pct','mean'),
    provided_sales_rows=('provided_sales_rows','sum'),retained_sales_rows=('retained_sales_rows','sum')).reset_index()
density_bands['complete_area_quarter_pct']=100*density_bands.complete_area_quarters/density_bands.area_quarters
density_bands['retained_of_provided_sales_rows_pct']=100*density_bands.retained_sales_rows/density_bands.provided_sales_rows.replace(0,np.nan)
display(density_bands)

,candidate_density_band,area_quarters,complete_area_quarters,mean_availability_pct,provided_sales_rows,retained_sales_rows,complete_area_quarter_pct,retained_of_provided_sales_rows_pct
0,0,535,535,100.000000,1869,1869,100.000000,100.000000
1,1-9,519,423,92.690002,2244,1793,81.502890,79.901961
2,10-49,536,290,96.201450,2277,1267,54.104478,55.643390
3,50-99,88,20,95.795087,425,89,22.727273,20.941176
4,100+,134,3,94.219587,846,24,2.238806,2.836879


**가능한 선택 영향:** 후보가 많은 상권·분기는 후보 중 단 하나라도 미가용이면 100% 기준에서 빠진다. 아래 차이는 완전 가용 표본이 공연 밀집 상권을 얼마나 대표하는지 판단하는 근거다. 누락이 매출·효과와 어떤 관련이 있는지 또는 추정치 편향의 방향은 이 기술표로 알 수 없다.

## 4. 자치구별 제공률과 100% 규칙의 추가 제외
원 매출 제공률과 일정 가용률로 남는 비율을 분리한다. 자치구별 남은 표본 비중은 원 모집단의 대표성을 보장하지 않는다.

In [5]:
district_rows=[]
for district,full in panel.groupby('district'):
    provided=full.loc[full.total_sales.gt(0)&full.night_sales.notna()&full.night_sales.ge(0)]
    kept=provided.loc[provided.availability_pct_centroid500.ge(100)]
    dqa=aq.loc[aq.district.eq(district)]
    positive=dqa.loc[dqa.candidate_id_count.gt(0)]
    district_rows.append(dict(district=district,all_panel_rows=len(full),all_areas=int(full.area_id.nunique()),
        original_provided_sales_rows=len(provided),original_sales_provision_pct=100*len(provided)/len(full),
        retained_sales_rows=len(kept),retained_of_provided_sales_rows_pct=100*len(kept)/len(provided),
        retained_of_all_panel_rows_pct=100*len(kept)/len(full),retained_areas=int(kept.area_id.nunique()),
        area_quarters=len(dqa),complete_area_quarters=int(dqa.complete_schedule_coverage.sum()),
        positive_candidate_area_quarters=len(positive),complete_positive_candidate_area_quarters=int(positive.complete_schedule_coverage.sum()),
        complete_positive_candidate_area_quarter_pct=100*positive.complete_schedule_coverage.mean(),
        zero_candidate_retained_sales_rows=int(kept.zero_registered_candidate_centroid500.sum())))
districts=pd.DataFrame(district_rows)
display(districts)
area_selection=aq.groupby(['area_id','area_name','district']).agg(area_quarters=('quarter','size'),
    complete_area_quarters=('complete_schedule_coverage','sum'),candidate_id_count_mean=('candidate_id_count','mean'),
    candidate_id_count_max=('candidate_id_count','max'),known_scheduled_start_count_mean=('known_scheduled_start_count','mean'),
    provided_sales_rows=('provided_sales_rows','sum'),retained_sales_rows=('retained_sales_rows','sum')).reset_index()
area_selection['retained_of_provided_sales_rows_pct']=100*area_selection.retained_sales_rows/area_selection.provided_sales_rows.replace(0,np.nan)
display(area_selection.sort_values('candidate_id_count_mean',ascending=False).head(10))

,district,all_panel_rows,all_areas,original_provided_sales_rows,original_sales_provision_pct,retained_sales_rows,retained_of_provided_sales_rows_pct,retained_of_all_panel_rows_pct,retained_areas,area_quarters,complete_area_quarters,positive_candidate_area_quarters,complete_positive_candidate_area_quarters,complete_positive_candidate_area_quarter_pct,zero_candidate_retained_sales_rows
0,마포구,9240,77,4326,46.818182,3077,71.128063,33.300866,67,924,702,629,407,64.705882,1230
1,종로구,8880,74,3335,37.556306,1965,58.920540,22.128378,51,888,569,648,329,50.771605,639


,area_id,area_name,district,area_quarters,complete_area_quarters,candidate_id_count_mean,candidate_id_count_max,known_scheduled_start_count_mean,provided_sales_rows,retained_sales_rows,retained_of_provided_sales_rows_pct
101,3120012,대학로(혜화역),종로구,12,0,399.000000,545,6958.833333,120,0,0.000000
27,3110027,혜화역 1번,종로구,12,0,297.750000,340,6048.750000,48,0,0.000000
23,3110023,서울대병원,종로구,12,0,265.500000,308,5458.166667,24,0,0.000000
112,3120103,홍대입구역(홍대),마포구,12,0,240.250000,353,459.083333,120,0,0.000000
142,3130190,홍대소상공인상점가,마포구,12,0,202.333333,312,417.833333,91,0,0.000000
24,3110024,혜회동주민센터,종로구,12,0,197.666667,326,2881.833333,27,0,0.000000
141,3130189,서교시장,마포구,12,0,192.250000,278,408.666667,12,0,0.000000
21,3110021,성균관대학교,종로구,12,0,143.916667,250,2076.166667,83,0,0.000000
114,3120105,상수역(홍대),마포구,12,1,141.416667,251,220.500000,109,9,8.256881
111,3120102,서교동(홍대),마포구,12,0,138.500000,193,506.833333,115,0,0.000000


## 5. 저장과 해석 범위
산출물은 `reports/final/revised_quarterly/14_*`로만 저장한다. 이 진단의 후보 수·예정 시작·자치구 제공률은 표본 구성의 관측된 차이다. 선택 편향의 크기나 효과의 부호를 계산한 결과가 아니다. 업종별 최소 군집 필터 적용 후 최종 추정 표본은 13번 결과에서 별도로 확인한다.

In [6]:
outputs={
    '14_zero_exposure_selection.csv':selection,'14_entity_observation_audit.csv':entities,
    '14_density_comparison.csv':density,'14_candidate_density_bands.csv':density_bands,
    '14_district_selection.csv':districts,'14_area_selection.csv':area_selection}
records={}
for name,frame in outputs.items():
    path=OUT/name;frame.to_csv(path,index=False,encoding='utf-8-sig')
    records[path.relative_to(ROOT).as_posix()]={'rows':len(frame),'columns':frame.columns.tolist(),'sha256':hashlib.sha256(path.read_bytes()).hexdigest()}
for relative,record in inputs.items():
    record['sha256_after']=hashlib.sha256((ROOT/relative).read_bytes()).hexdigest()
    record['unchanged']=record['sha256_before']==record['sha256_after']
assert all(row['unchanged'] for row in inputs.values())
summary={'status':'complete','created_utc':pd.Timestamp.now(tz='UTC').isoformat(),
    'source_dataset_summary':'reports/final/revised_quarterly/12_dataset_summary.json',
    'plan_sha256':summary12['plan_sha256'],'all_inputs_unchanged':True,'new_regression_fits':0,'p_values_examined':False,
    'network_requests':0,'spatial_definition':'centroid500','availability_threshold_pct':100,
    'before_industry_minimum_cluster_filter':True,'selection':selection.to_dict('records'),
    'density_comparison':density.to_dict('records'),'candidate_density_bands':density_bands.to_dict('records'),
    'district_selection':districts.to_dict('records'),
    'scope_notes':['zero_registered_candidate is restricted to fixed observed official-ID candidate pool',
        'all-zero entities refer only to each selected sample observed quarters, not unobserved actual performances',
        'known scheduled counts in incomplete groups are partial, not true sessions or audience',
        'complete-ID selection can differ by candidate density and district; causal bias direction and magnitude are not identified',
        'group tables are descriptive; no new coefficients, p values or outcome association were calculated'],
    'inputs':list(inputs.values()),'outputs':records}
(OUT/'14_summary.json').write_text(json.dumps(summary,ensure_ascii=False,indent=2),encoding='utf-8')
print(f'완료: CSV {len(outputs)}개, 14_summary.json, 입력 {len(inputs)}개 불변')
display(selection[['sample','sales_rows','areas','zero_candidate_sales_row_pct','entities_all_selected_observations_zero','sales_rows_in_all_zero_entities_pct']])

완료: CSV 6개, 14_summary.json, 입력 37개 불변


,sample,sales_rows,areas,zero_candidate_sales_row_pct,entities_all_selected_observations_zero,sales_rows_in_all_zero_entities_pct
0,common_outcome_before_repeat_filter,5042,118,37.068624,59,10.313368
1,repeat_ge2_before_min_cluster_filter,5015,114,37.148554,53,10.249252


**최종 해석:** 주 결과를 설명할 때 전체 종로·마포 외식업을 대표하는 효과라고 일반화하지 않는다. 매출 제공과 엄격 일정 가용률을 함께 충족한 상권·업종·분기에서 관측한 관련성이라는 범위를 유지한다. 더 넓은 표본의 민감도 결과가 있어도 같은 선택 문제가 사라졌다고 단정하지 않는다.